In [3]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf

In [2]:
from google.colab import files

uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]

print("Uploaded file:", pdf_name)

Saving Assignment4 AIML 23btce105.pdf to Assignment4 AIML 23btce105 (1).pdf
Uploaded file: Assignment4 AIML 23btce105 (1).pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF pages
pages = PyPDFLoader(pdf_name).load()

print("Number of pages:", len(pages))

# Split PDF text into smaller chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Number of chunks:", len(chunks))

/tmp/ipykernel_3922/3674446095.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 6
Number of chunks: 12


In [1]:
!pip install -q langchain-huggingface sentence-transformers

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Create free local embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store document chunks in Chroma
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="lab4_documents"
)

print("Vector database ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database ready!


In [7]:
import os
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM is ready!")

Groq LLM is ready!


In [10]:
import langchain

print(langchain.__version__)

1.3.13


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

retriever = db.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the document context below.

If the answer is not available in the context, say:
"The answer is not available in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
""")

chain = prompt | llm | StrOutputParser()

def ask_document(question):
    documents = retriever.invoke(question)

    context = "\n\n".join(
        document.page_content for document in documents
    )

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    return answer

print("RAG system is ready!")

RAG system is ready!


In [12]:
question = "What is this document about?"

answer = ask_document(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: What is this document about?

Answer: This document is about Support Vector Machine (SVM), a supervised machine learning algorithm, as part of an assignment for the subject Artificial Intelligence And Machine Learning.
